# Does the direct `q''` estimator average stably?

**Question.** The direct tangent method gives each trajectory its own first and second
derivative of the reduced arrow-of-time density $q=\overline{1+z^2}$ with respect to
$\theta=\log\gamma$ at fixed $J$ (equivalently $\log g$, $g=\gamma/4J$). The reported
derivative is the average over independent trajectories. The tangents are correct:
common-noise finite differences converge to them at $O(h^2)$
(`docs/derivative/aot_tangent_growth_diagnosis.md`). This notebook asks whether
*averaging* them is statistically stable: does adding hundreds or thousands of
trajectories make the estimate settle?

**How to read the plots.** A well-defined average shows all of the following:

| Diagnostic | Stable (finite variance) | Unstable (heavy tail) |
|---|---|---|
| Per-trajectory values | a band of similar size | rare values orders of magnitude larger |
| Running mean with 95% CI | settles inside a shrinking band | jumps when a rare trajectory arrives |
| $\mathrm{SEM}\cdot\sqrt N$ | flat (central limit theorem) | keeps rising as $N$ grows |
| Survival $P(\lvert X\rvert\ge x)$, log-log | falls faster than slope $-2$ | straight line with slope $-\alpha$, $\alpha<2$ |
| Tail index $\alpha$ (Hill) | $\alpha>2$ | $\alpha<2$: infinite variance; $\alpha<1$: infinite mean |

**Protocol.** Generated by `scripts/run_tangent_sample_scaling.py` with the accepted serial
CPU estimators, no trimming, independent seeded chunks:

* **Chain:** periodic Ising chain, $L=8$, $J=1$, $dt=0.00125$, burn-in time 5, averaging
  time 10 (the frozen WBS 5.5 sentinel protocol), Gaussian noise, 2048 trajectories per $g$.
  The first 8 trajectories at each $g$ replay the sentinel exactly.
* **Single qubit:** $H=J\sigma_y$, monitored $\sigma_z$, same $J$, $dt$, burn-in and averaging
  time, Rademacher noise, 65,536 trajectories per $g$. A sampling-free stationary reference
  from the deterministic transfer operator is drawn where it is numerically usable.

Trajectories are shown in generation (seed) order; nothing is sorted or selected.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import SymmetricalLogLocator

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quantum_measurement.utilities.heavy_tail import (
    bootstrap_hill_interval,
    running_max_share,
    running_mean_sem,
    survival_function,
)

RESULTS = Path(os.environ.get('AOT_SCALING_RESULTS', ROOT / 'results'))
RUNS = {
    'chain L=8': RESULTS / 'aot_tangent_sample_scaling_chain',
    'single qubit': RESULTS / 'aot_tangent_sample_scaling_single_qubit',
}
FIGURES = RESULTS / 'aot_tangent_sample_scaling_figures'
FIGURES.mkdir(parents=True, exist_ok=True)
FIELDS = ('q', 'dq_dtheta', 'd2q_dtheta2', 'max_first_tangent_norm', 'max_second_tangent_norm')
LABELS = {'q': "$q$", 'dq_dtheta': "$q'$", 'd2q_dtheta2': "$q''$"}
HILL_FRACTION = 0.05
BOOTSTRAP = 200
SEM_REFERENCE_N = 128
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_run(directory):
    # Load every chunk in seed order and verify its recorded checksum.
    manifest = json.loads((directory / 'manifest.json').read_text())
    if manifest['status'] != 'complete':
        raise RuntimeError(f'{directory} is not complete')
    config = manifest['configuration']
    data = {}
    for g_index, g in enumerate(config['g_values']):
        entries = sorted(
            (entry for entry in manifest['chunks'].values() if entry['g_index'] == g_index),
            key=lambda entry: entry['chunk'],
        )
        arrays = []
        for entry in entries:
            path = directory / 'chunks' / entry['file']
            if sha256(path) != entry['sha256']:
                raise RuntimeError(f'checksum mismatch: {path}')
            arrays.append(np.load(path))
        data[g] = {field: np.concatenate([a[field] for a in arrays]) for field in FIELDS}
    reference = None
    if (directory / 'reference.json').exists():
        reference = json.loads((directory / 'reference.json').read_text())
    return config, manifest, data, reference


runs = {name: load_run(path) for name, path in RUNS.items()}
for name, (config, manifest, data, _) in runs.items():
    counts = {g: len(values['q']) for g, values in data.items()}
    print(f"{name}: {counts} trajectories; commit {manifest['environment']['git']['commit'][:8]}"
          f" (dirty={manifest['environment']['git']['dirty']})")

## 1. Replay check

The first 8 chain trajectories at each $g$ use the WBS 5.5 sentinel noise, so they must
reproduce the tangent-growth diagnosis exactly. A mismatch would invalidate the comparison.

In [ ]:
diagnosis = {0.1: 'aot_tangent_growth_g0p1', 1.0: 'aot_tangent_growth_g1', 10.0: 'aot_tangent_growth_g10'}
chain_data = runs['chain L=8'][2]
for g, folder in diagnosis.items():
    path = RESULTS / folder / 'tangent_growth_results.json'
    if not path.exists():
        print(f'g={g:g}: diagnosis output not found, replay check skipped')
        continue
    expected = np.array(json.loads(path.read_text())['primary']['d2q']['values'])
    observed = chain_data[g]['d2q_dtheta2'][:expected.size]
    print(f'g={g:g}: max |difference| over the 8 sentinel trajectories = '
          f'{np.max(np.abs(observed - expected)):.3g}')

## 2. Summary table

For each system, $g$ and quantity: the trajectory mean and SEM, the median, the Hill tail
index of $|X|$ from the largest 5% with a 95% bootstrap interval, the share of the single
largest $|X_i|$ in $\sum_i|X_i|$, and the **SEM scaling ratio**
$\mathrm{SEM}(N)\sqrt{N}\,/\,\mathrm{SEM}(128)\sqrt{128}$. Under the central limit theorem
the ratio is close to 1; values well above 1 mean the error bar is not shrinking like
$1/\sqrt N$. The Hill index is not meaningful for the bounded base observable $q\in[1,2]$.

In [ ]:
def summarize(values, field, rng):
    counts, means, sems = running_mean_sem(values)
    reference_index = min(SEM_REFERENCE_N, counts[-1]) - 2
    ratio = (sems[-1] * np.sqrt(counts[-1])) / (sems[reference_index] * np.sqrt(counts[reference_index]))
    row = {
        'N': int(counts[-1]),
        'mean': means[-1],
        'sem': sems[-1],
        'median': float(np.median(values)),
        'max share': float(running_max_share(values)[-1]),
        'SEM scaling ratio': float(ratio),
    }
    if field != 'q':
        alpha, low, high = bootstrap_hill_interval(np.abs(values), HILL_FRACTION, BOOTSTRAP, rng)
        row.update({'Hill alpha': alpha, 'alpha 95% low': low, 'alpha 95% high': high})
    return row


rng = np.random.default_rng(20261004)
rows = []
for name, (config, _, data, _) in runs.items():
    for g, values in data.items():
        for field in ('q', 'dq_dtheta', 'd2q_dtheta2'):
            rows.append({'system': name, 'g': g, 'quantity': LABELS[field],
                         **summarize(values[field], field, rng)})
summary = pd.DataFrame(rows)
summary.to_csv(FIGURES / 'summary.csv', index=False)
with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 200):
    display(summary)

In [ ]:
reference_rows = []
reference = runs['single qubit'][3]
sq_data = runs['single qubit'][2]
if reference is not None:
    for point in reference['points']:
        preferred = point['preferred'] or {}
        values = sq_data[point['g']]
        rows_for = {}
        for field, key in (('q', 'q'), ('dq_dtheta', 'dq_dlog_g'), ('d2q_dtheta2', 'd2q_dlog_g2')):
            counts, means, sems = running_mean_sem(values[field])
            rows_for[field] = (means[-1], sems[-1], preferred.get(key, np.nan))
        reference_rows.append({
            'g': point['g'], 'reference usable': point['usable'],
            'sampled q': rows_for['q'][0], 'q SEM': rows_for['q'][1], 'stationary q': rows_for['q'][2],
            "sampled q'": rows_for['dq_dtheta'][0], "q' SEM": rows_for['dq_dtheta'][1],
            "stationary q'": rows_for['dq_dtheta'][2],
            "sampled q''": rows_for['d2q_dtheta2'][0], "q'' SEM": rows_for['d2q_dtheta2'][1],
            "stationary q''": rows_for['d2q_dtheta2'][2],
        })
reference_table = pd.DataFrame(reference_rows)
with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 200):
    display(reference_table)

## 3. Headline figure: chain $L=8$, $q''$

Columns are $g=0.1, 1, 10$. Rows: per-trajectory values (symmetric-log axis); running mean
with 95% CI; $\mathrm{SEM}\cdot\sqrt N$; survival function of $|q''|$ with reference slopes
$\alpha=1$ and $\alpha=2$ and the Hill estimate.

In [ ]:
def stability_figure(name, field, title, reference=None):
    config, _, data, _ = runs[name]
    g_values = list(data)
    fig, axes = plt.subplots(4, len(g_values), figsize=(4.6 * len(g_values), 13), squeeze=False)
    rng = np.random.default_rng(7)
    for column, g in enumerate(g_values):
        values = data[g][field]
        counts, means, sems = running_mean_sem(values)
        scale = max(float(np.median(np.abs(values))), 1e-12)

        ax = axes[0, column]
        ax.plot(np.arange(1, values.size + 1), values, '.', ms=2, alpha=0.6)
        ax.set_yscale('symlog', linthresh=scale)
        decades = np.log10(max(float(np.max(np.abs(values))), scale) / scale)
        ax.yaxis.set_major_locator(SymmetricalLogLocator(
            linthresh=scale, base=10.0 ** max(1, int(np.ceil(decades / 5.0)))))
        ax.set_title(f'{title}, g = {g:g}')
        ax.set_xlabel('trajectory index (seed order)')
        ax.set_ylabel(f'per-trajectory {LABELS[field]}')

        ax = axes[1, column]
        ax.fill_between(counts, means - 1.96 * sems, means + 1.96 * sems, alpha=0.3, label='95% CI')
        ax.plot(counts, means, lw=1.2, label='running mean')
        if reference is not None and reference.get(g) is not None:
            ax.axhline(reference[g], color='black', ls='--', lw=1, label='stationary reference')
        ax.set_xscale('log')
        ax.set_xlabel('number of trajectories N')
        ax.set_ylabel(f'mean {LABELS[field]}')
        ax.legend(fontsize=8)

        ax = axes[2, column]
        ax.loglog(counts, sems * np.sqrt(counts))
        ax.set_xlabel('number of trajectories N')
        ax.set_ylabel(r'SEM $\times\sqrt{N}$ (flat if CLT holds)')

        ax = axes[3, column]
        magnitudes, survival = survival_function(np.abs(values))
        positive = magnitudes > 0
        ax.loglog(magnitudes[positive], survival[positive], lw=1.2, label='empirical')
        anchor = np.quantile(magnitudes[positive], 0.95)
        span = np.geomspace(anchor, magnitudes.max(), 20)
        for alpha, style in ((1.0, ':'), (2.0, '--')):
            ax.loglog(span, 0.05 * (span / anchor) ** (-alpha), 'k' + style, lw=1,
                      label=rf'slope $-{alpha:g}$')
        estimate, low, high = bootstrap_hill_interval(np.abs(values), HILL_FRACTION, BOOTSTRAP, rng)
        ax.text(0.03, 0.05, rf'Hill $\alpha$ = {estimate:.2f} [{low:.2f}, {high:.2f}]',
                transform=ax.transAxes, fontsize=9)
        ax.set_xlabel(f'|{LABELS[field]}|')
        ax.set_ylabel(r'$P(|X| \geq x)$')
        ax.legend(fontsize=8, loc='upper right')
    fig.tight_layout()
    return fig


fig = stability_figure('chain L=8', 'd2q_dtheta2', "chain L=8: q''")
fig.savefig(FIGURES / 'chain_d2q_stability.png', dpi=150)
fig.savefig(FIGURES / 'chain_d2q_stability.pdf')
plt.show()

## 4. Contrast: which quantities are well behaved?

Running mean with 95% CI and $\mathrm{SEM}\cdot\sqrt N$ for $q$, $q'$ and $q''$ at every $g$.

In [ ]:
def contrast_figure(name, reference=None):
    _, _, data, _ = runs[name]
    fields = ('q', 'dq_dtheta', 'd2q_dtheta2')
    g_values = list(data)
    fig, axes = plt.subplots(2 * len(g_values), 3, figsize=(14, 4.2 * len(g_values)), squeeze=False)
    for block, g in enumerate(g_values):
        for column, field in enumerate(fields):
            counts, means, sems = running_mean_sem(data[g][field])
            ax = axes[2 * block, column]
            ax.fill_between(counts, means - 1.96 * sems, means + 1.96 * sems, alpha=0.3)
            ax.plot(counts, means, lw=1.2)
            if reference is not None and reference.get((g, field)) is not None:
                ax.axhline(reference[(g, field)], color='black', ls='--', lw=1)
            ax.set_xscale('log')
            ax.set_title(f'{name}, g = {g:g}: {LABELS[field]}')
            ax.set_ylabel('running mean, 95% CI')
            ax = axes[2 * block + 1, column]
            ax.loglog(counts, sems * np.sqrt(counts))
            ax.set_ylabel(r'SEM $\times\sqrt{N}$')
            ax.set_xlabel('number of trajectories N')
    fig.tight_layout()
    return fig


fig = contrast_figure('chain L=8')
fig.savefig(FIGURES / 'chain_contrast.png', dpi=150)
fig.savefig(FIGURES / 'chain_contrast.pdf')
plt.show()

## 5. Single qubit (gauge-free control)

The same diagnostics for the driven single qubit. Dashed lines are the sampling-free
stationary reference wherever all grid and step-size choices agree (`reference usable`
in the table above); unusable reference points are not drawn.

In [ ]:
stationary = {}
stationary_contrast = {}
if reference is not None:
    for point in reference['points']:
        if point['usable']:
            preferred = point['preferred']
            stationary[point['g']] = preferred['d2q_dlog_g2']
            for field, key in (('q', 'q'), ('dq_dtheta', 'dq_dlog_g'), ('d2q_dtheta2', 'd2q_dlog_g2')):
                stationary_contrast[(point['g'], field)] = preferred[key]

fig = stability_figure('single qubit', 'd2q_dtheta2', "single qubit: q''", reference=stationary)
fig.savefig(FIGURES / 'single_qubit_d2q_stability.png', dpi=150)
fig.savefig(FIGURES / 'single_qubit_d2q_stability.pdf')
plt.show()

fig = contrast_figure('single qubit', reference=stationary_contrast)
fig.savefig(FIGURES / 'single_qubit_contrast.png', dpi=150)
fig.savefig(FIGURES / 'single_qubit_contrast.pdf')
plt.show()

## 6. Conclusions

Numbers below are from the summary tables above (run of 2026-10-04, commit `415c2f0` plus
uncommitted study scripts).

**1. More trajectories do not stabilize $q''$; near $g\approx1$ they make it worse.**

* *Chain $L=8$, $g=1$, 2048 trajectories:* $q''=8.4\times10^{7}\pm9.6\times10^{7}$ while the
  median is $-16.6$. A single trajectory with $|q''|\approx2\times10^{11}$ holds 88% of
  $\sum_i|q''_i|$. $\mathrm{SEM}\sqrt N$ is 322 times its value at $N=128$ instead of staying
  flat, and the tail index is $\alpha=0.35$ $[0.29, 0.43]$.
* *Single qubit, $g=1$, 65,536 trajectories:* $q''=-3.0\times10^{5}\pm3.1\times10^{5}$,
  $\alpha=0.51$ $[0.49, 0.53]$, while the sampling-free stationary value is $+0.084$.
* *$g=10$:* the same behaviour in both systems ($\alpha\approx0.5$).

With $\alpha<1$ over the sampled range, the sample mean is set by the largest trajectory
seen so far, and adding trajectories brings in larger ones. A finite run must have a cutoff
somewhere, but it lies beyond the $10^{11}$ already sampled, so no practical $N$ averages it
out. This is not a lack of trajectories; the per-trajectory estimator has no usable mean.

**2. The ensemble response itself is small and smooth.** The stationary single-qubit
reference gives $q''=0.014,\ 0.084,\ -0.054$ at $g=0.1, 1, 10$. The physics is fine; the
failure is in averaging pathwise derivatives, i.e. the derivative and the average cannot be
interchanged in practice for this protocol.

**3. $q$ is perfectly stable everywhere** ($\mathrm{SEM}\sqrt N$ ratio $\approx1$, largest
share $\le5\times10^{-4}$).

**4. $q'$ is also affected.** Only the chain at $g=0.1$ is healthy ($\alpha=3.05$). At chain
$g=1$, $q'=6.7\pm5.6$ against a median of $0.28$ with $\alpha=0.64$; at $g=10$ and for the
single qubit at $g=1, 10$, $\alpha\approx1.1$. The direct first-derivative estimator is
therefore also unreliable near and above $g\approx1$ for this protocol.

**5. Weak measurement ($g=0.1$) is borderline for the chain and unstable for the single
qubit.** The chain $q''$ has $\alpha=1.44$ $[1.23, 1.88]$ (summary table; the figure's independent
bootstrap gives a similar interval; formally infinite variance) but
no trajectory dominates at this $N$ (largest share 1.8%) and the running mean settles near
$0.02\pm0.02$. The single qubit at $g=0.1$ has $\alpha=1.01$ $[0.98, 1.04]$ and
$\mathrm{SEM}\sqrt N$ grows ninefold from $N=128$ to $65{,}536$; its running mean wanders
by $\pm0.2$ around the stationary value $0.014$ before ending near it.

**6. Protocol note.** For the single qubit at $g=0.1$ the burn-in of 5 time units is too
short: sampled $q$ is 16 SEM above the stationary value. A separate check with 16,384
trajectories gave $+8.1$ SEM at burn-in 5 and $-1.0$ SEM at burn-in 20. This biases the
base $q$ slightly and does not affect the tail conclusions. The chain burn-in was validated
separately (`aot_gaussian_orbital_stationarity.md`).

**Implication.** Derivative fields for M3 cannot come from the direct pathwise estimator at
this protocol near $g\gtrsim1$, regardless of trajectory count. A different estimator (for
example a likelihood-ratio / score-function estimator) is needed, validated first on the
single qubit against the stationary reference.

**Caveats.** Hill estimates use the largest 5% of $|X|$; the survival plots show the tail
over several decades directly. The tail exponent is measured for $T_{\rm avg}=10$ after a
burn-in of 5 and may depend on the horizon. Reproduce with the commands in
`docs/derivative/aot_tangent_sample_scaling.md`.